In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Qdrant
from langchain.chains.query_constructor.base import StructuredQueryOutputParser, get_query_constructor
from langchain.retrievers.self_query.base import SelfQueryRetriever
from app.utils.settings import COLLECTION_NAME, QDRANT_HOST, QDRANT_PORT
from app.retrieval.self_query import metadata_field_info, document_content_description
# from langchain_openai import OpenAI

def get_retriever(llm):
    embeddings_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    qdrant = Qdrant.from_existing_collection(
        embedding=embeddings_model,
        collection_name=COLLECTION_NAME,
        url=f"http://{QDRANT_HOST}:{QDRANT_PORT}"
    )
    
    query_constructor = get_query_constructor(
        prompt=None,  # Use default ou customize
        metadata_field_info=metadata_field_info,
        document_contents=document_content_description,
    )
    query_parser = StructuredQueryOutputParser.from_components()
    
    retriever = SelfQueryRetriever.from_llm(
        llm=llm,
        vectorstore=qdrant,
        query_constructor=query_constructor,
        structured_query_translator=query_parser,
        verbose=True
    )
    return retriever